The following file is used to create the results from section 4.1 in the paper. We train 4 metanetworks, one for each dataset and evaluate performance on the validation set.

In [ ]:
DATASETS = ['mnist', 'fashion_mnist', 'cifar10', 'svhn_cropped']

In [ ]:
from cnn_surgery.utils.load_dataset import load_multi_stage_dataset, load_dataset
from cnn_surgery.lenses.regressor_lens import get_regressor_lens, mse_mae
import pickle
N = 10  # number of metanetworks to train per dataset

In [4]:
results = {}
for dataset in DATASETS:
    results[dataset] = []
    for i in range(N):  # Train N metanetworks per dataset
        data = load_multi_stage_dataset(dataset=dataset)
        weights_train, accuracies_train, config_train = data['train']
        weights_val, accuracies_val, config_val = data['val']
        meta_network, metrics = get_regressor_lens(weights_train, accuracies_train, weights_val, accuracies_val, device='cpu', verbose=True, return_metrics=True)
        ((mse_train, mae_train), (mse_val, mae_val), r2) = metrics
        results[dataset].append({
            'mse_train': mse_train,
            'mae_train': mae_train,
            'mse_val': mse_val,
            'mae_val': mae_val,
            'r2': r2
        })
        # Save each metanetwork for later use
        pickle.dump(meta_network, open(f'meta_network_{dataset}_{i}.pkl', 'wb'))

In [ ]:
import pandas as pd
print(pd.DataFrame(results).T.to_latex(float_format="%.3f"))

In [12]:
import pandas as pd
df_results = pd.DataFrame.from_dict({
    (dataset, i): metrics
    for dataset, dataset_results in results.items()
    for i, metrics in enumerate(dataset_results)
}, orient='index')

df_results.index = pd.MultiIndex.from_tuples(df_results.index, names=["Dataset", "Model"])
df_results.columns = ["mse_train", "mae_train", "mse_val", "mae_val", "r2"]
display(df_results)

mse_train  mae_train   mse_val   mae_val        r2
Dataset       Model                                                    
mnist         0       0.011897   0.049365  0.014034  0.055919  0.921726
              1       0.011566   0.048584  0.013673  0.055083  0.923741
              2       0.011732   0.050037  0.013752  0.056265  0.923302
              3       0.012027   0.050619  0.013850  0.056376  0.922755
              4       0.012080   0.049397  0.014341  0.056040  0.920019
              5       0.011828   0.049395  0.013919  0.055954  0.922372
              6       0.011982   0.049760  0.013974  0.056176  0.922062
              7       0.012079   0.049781  0.014008  0.055888  0.921873
              8       0.012158   0.050917  0.013961  0.056733  0.922135
              9       0.011859   0.049245  0.013910  0.055444  0.922420
fashion_mnist 0       0.012814   0.060447  0.015649  0.068110  0.894677
              1       0.013198   0.064214  0.016626  0.072965  0.888099
              2       0.012624   0.060416  0.015713  0.068418  0.894246
              3       0.012854   0.060081  0.016038  0.068197  0.892060
              4       0.013572   0.061592  0.017051  0.070091  0.885239
              5       0.012976   0.060506  0.015933  0.068538  0.892768
              6       0.012690   0.059701  0.015851  0.067586  0.893318
              7       0.012159   0.058486  0.015508  0.067176  0.895627
              8       0.012103   0.057966  0.015530  0.066700  0.895477
              9       0.012748   0.059422  0.015738  0.067386  0.894080
cifar10       0       0.011079   0.059909  0.014922  0.074454  0.705018
              1       0.011046   0.060250  0.014829  0.074482  0.706858
              2       0.011788   0.062778  0.015359  0.075883  0.696365
              3       0.012149   0.063481  0.015862  0.076946  0.686433
              4       0.010944   0.060079  0.014860  0.074579  0.706234
              5       0.013025   0.063780  0.016680  0.077814  0.670256
              6       0.011522   0.061066  0.015209  0.074853  0.699339
              7       0.012388   0.063797  0.016038  0.077611  0.682948
              8       0.011339   0.061135  0.015178  0.075318  0.699949
              9       0.011281   0.061878  0.014951  0.075362  0.704444
svhn_cropped  0       0.008616   0.044688  0.011162  0.053437  0.864711
              1       0.015414   0.056870  0.017899  0.065076  0.783047
              2       0.008788   0.046149  0.011305  0.054629  0.862975
              3       0.008751   0.045727  0.011366  0.054480  0.862241
              4       0.009424   0.047076  0.012049  0.055889  0.853959
              5       0.008945   0.046337  0.011360  0.054405  0.862308
              6       0.008714   0.045401  0.011276  0.053959  0.863320
              7       0.008636   0.045405  0.011102  0.053890  0.865438
              8       0.008735   0.045658  0.011205  0.054372  0.864183
              9       0.008611   0.044688  0.011239  0.053606  0.863778

In [22]:
grouped_mean = df_results.groupby("Dataset").mean()
grouped_std = df_results.groupby("Dataset").std()

combined = grouped_mean.copy()
for col in combined.columns:
    combined[col] = grouped_mean[col].map(lambda x: f"{x:.3f}") + r" $\pm$ " + grouped_std[col].map(lambda x: f"{x:.4f}")

# re-order in DATASETS order
combined = combined.reindex(DATASETS)
print(combined.to_latex(escape=False))

\begin{tabular}{llllll}
\toprule
 & mse_train & mae_train & mse_val & mae_val & r2 \\
Dataset &  &  &  &  &  \\
\midrule
mnist & 0.012 $\pm$ 0.0002 & 0.050 $\pm$ 0.0007 & 0.014 $\pm$ 0.0002 & 0.056 $\pm$ 0.0005 & 0.922 $\pm$ 0.0010 \\
fashion_mnist & 0.013 $\pm$ 0.0004 & 0.060 $\pm$ 0.0017 & 0.016 $\pm$ 0.0005 & 0.069 $\pm$ 0.0018 & 0.893 $\pm$ 0.0034 \\
cifar10 & 0.012 $\pm$ 0.0007 & 0.062 $\pm$ 0.0015 & 0.015 $\pm$ 0.0006 & 0.076 $\pm$ 0.0013 & 0.696 $\pm$ 0.0121 \\
svhn_cropped & 0.009 $\pm$ 0.0021 & 0.047 $\pm$ 0.0036 & 0.012 $\pm$ 0.0021 & 0.055 $\pm$ 0.0035 & 0.855 $\pm$ 0.0253 \\
\bottomrule
\end{tabular}

